# 01 – Clean & explore
Input: `data_raw/appliance_brand_reviews.csv`  
Output: `data_clean/reviews_laundry.csv`

Change `CATEGORY` to `refrigerator_freezer` to replicate on the backup category.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

CATEGORY = "laundry"          # backup: "refrigerator_freezer"
RAW = "data_raw/appliance_brand_reviews.csv"
OUT = "data_clean/reviews_laundry.csv"   # notebook 02 reads this exact name

In [ ]:
df = pd.read_csv(RAW, parse_dates=["review_date"])
print(df.shape)
df.head()

## Coverage check (why we chose the category)

In [ ]:
print(pd.crosstab(df.product_category, df.brand))
print(df.groupby("product_category").review_rating.mean().round(2))

## Filter to category and clean

In [ ]:
d = df[df.product_category == CATEGORY].copy()
d = d.drop_duplicates("review_id")
d["review_text"] = d.review_text.fillna("")
d["review_title"] = d.review_title.fillna("")
d["text"] = (d.review_title + ". " + d.review_text).str.strip()
d["text"] = d.text.str.replace(r"<[^>]+>|&#?\w+;", " ", regex=True).str.replace(r"\s+", " ", regex=True)  # strip <br />, &#34; etc.
d = d[d.text.str.len() > 5]
d["n_words"] = d.text.str.split().str.len()
d["year"] = d.review_date.dt.year
d["is_lg"] = (d.brand == "LG").astype(int)

# Accessories/parts check: eyeball titles with these words, then drop if they are not appliances
PARTS = r"(?:filter|gasket|hose|cover|rack|knob|part|replacement|kit|cleaner|tablet|detergent)"
sus = d[d.product_title.str.contains(PARTS, case=False, regex=True)]
print("suspicious titles:", sus.product_id.nunique())
sus.drop_duplicates("product_id").product_title.head(30).tolist()

In [ ]:
# Uncomment after reading the list above
# d = d[~d.product_id.isin(sus.product_id)]

# Portable/compact washers (almost all Haier, ~4.1 stars) are a different segment from LG's full-size machines
d = d[~d.product_title.str.contains("portable|compact", case=False)]

# Low rating definition (see PLAN.md 3.2). 3-star dropped from the classification sample.
d["low"] = np.where(d.review_rating <= 2, 1, np.where(d.review_rating >= 4, 0, np.nan))
d["low_strict"] = (d.review_rating == 1).astype(int)   # sensitivity check
print(d.shape, "| 3-star share:", round((d.review_rating == 3).mean(), 3))

## Descriptives
**Check:** Haier laundry reviews are mostly *portable* washers (a different segment, mean ~4.1 stars). Excluding Haier from the competitor set is recommended (see next cell).

In [ ]:
d[d.brand == "Haier"].product_title.value_counts().head(10)

In [ ]:
print(d.brand.value_counts())
print(d.groupby("brand").agg(n=("review_id", "size"), mean_rating=("review_rating", "mean"), low_share=("low", "mean")).round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=d, x="review_rating", ax=ax[0]); ax[0].set_title("Rating distribution (polarised)")
sns.barplot(data=d, x="brand", y="low", order=d.groupby("brand").low.mean().sort_values().index, ax=ax[1])
ax[1].set_title("Share of low (1-2 star) reviews by brand"); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig("figures/01_ratings_by_brand.png", dpi=150)

In [ ]:
d.groupby("year").agg(n=("review_id", "size"), low=("low", "mean")).round(3)

In [ ]:
d.to_csv(OUT, index=False)
print("saved", OUT, d.shape)